# 🧮 How the improved system calculates its results

This notebook explains what happens inside the **improved (condition-aware)** system

---

## 🤖 formulas and rules

The improved system uses **no AI**: no ChatGPT, no language model, no trained machine-learning model. It consists only of **formulas and fixed rules**, so every result can be recalculated by hand.

| Component | Technique |  |
|---|---|---|
| Text search | BM25, a statistical formula (library `rank_bm25`) |  |
| Recognising places despite typos | String similarity (library `rapidfuzz`) |  |
| Dates such as *"20 Oct"* | Date parser (library `dateparser`) |  |
| Skill level, time, regions | Fixed word lists and rules |  |
| Scoring the conditions | Formulas using the thresholds in `surfability_thresholds.csv` |  |
| Explanation sentence | Text template filled with the computed numbers |  |
| *Optional:* dense search | Small embedding model `bge-small-en-v1.5` | switched off by default |



## 1️⃣ Understanding the query (rules)

- **Place:** the query is split into word groups of 1–3 words. Each group is compared with all place and spot names; a similarity of **at least 88 %** counts as a match, so typos such as *"Kaohsuing"* still work.
- **"near X":** all spots within **40 km** (straight-line distance, haversine formula) remain. Region phrases such as *"east coast"* keep the spots in those regions.
- **Skill level and time:** pattern matching in the text, e.g. *"tomorrow"* = day + 1, *"morning"* = 06:00–11:00, counted from the moment the snapshot was taken.

## 2️⃣ Text score (BM25)

For every word $t$ in the query, each spot description $d$ earns points:

$$
\text{points}(t, d) = \text{IDF}(t) \cdot \frac{f_{t,d} \cdot (k_1 + 1)}{f_{t,d} + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avg}|d|}\right)}
\qquad
\text{IDF}(t) = \log\left(1 + \frac{N - n_t + 0.5}{n_t + 0.5}\right)
$$

| Symbol | Meaning | Value |
|---|---|---|
| $f_{t,d}$ | how often word $t$ appears in description $d$ | – |
| $n_t$ | number of spots whose description contains $t$ | – |
| $N$ | number of spots | 35 |
| $\lvert d\rvert$ / $\text{avg}\lvert d\rvert$ | description length / average length (length correction) | – |
| $k_1$ | how quickly repeated words stop adding points | 1.2 |
| $b$ | strength of the length correction | 0.75 |

**Intuition:**
- **Rare words count more:** *reef* earns more than *break*, which appears almost everywhere.
- **Repeated words** earn more points, but with diminishing returns.
- **Long descriptions** are scored down slightly.

**Slang expansions** (e.g. *newbie* → *beginner*) are added with **weight 0.5**. The sum is divided by the highest score of all spots, so the **text score lies between 0 and 1**.

## 3️⃣ Condition fit (scoring the conditions)

For **every hour** in the time window, six factors between 0 and 1 are calculated and **multiplied**:

$$
CF = \text{size} \times \text{period} \times \text{direction} \times \text{wind} \times \text{tide}^{0.5} \times \text{level}
$$

The **best hour** in the time window counts. Multiplying means a single knock-out factor makes the score low: perfect wind does not help when there are no waves.

### Worked example (illustration)

**Wai'ao, intermediate surfer.** Forecast: 0.8 m swell from the east, 5 s period, 7 km/h wind from the southeast, mid tide.

| Factor | Rule (thresholds for intermediate) | Calculation | Value |
|---|---|---|---|
| 🌊 **Size** | 0 below 0.6 m, rises to 1 at 1.0 m, stays 1 up to 1.5 m, falls to 0 at 1.8 m | (0.8 − 0.6) / (1.0 − 0.6) | **0.50** |
| ⏱️ **Period** | 0.4 at 6 s or less, rises to 1 at 9 s | 5 s ≤ 6 s | **0.40** |
| 🧭 **Direction** | 1 if the swell is inside the spot's window (60–160°) | east = 90°, inside | **1.00** |
| 💨 **Wind** | The beach faces 98°, so offshore wind would come from 278°. Southeast (135°) is onshore. Penalty = onshore share × wind strength | 1 − 0.87 × (7 − 5.5) / (28 − 5.5) | **0.94** |
| 🌙 **Tide** | 1 if the tide stage matches (Wai'ao: mid/high) | mid matches | **1.00** |
| 🏄 **Level** | 1 if the spot profile lists this skill level | intermediate is listed | **1.00** |

$$
CF = 0.50 \times 0.40 \times 1.00 \times 0.94 \times 1.00 \times 1.00 \approx 0.19 \;\Rightarrow\; \text{"Poor", explanation: "Limited by short period"}
$$

- The **smallest factor** (here the period) is named as the reason in the explanation.
- From **0.5** a spot counts as **surfable ("Fair")**, from **0.75** as **"Good"**.

### Correction by buoy observations

If a nearby buoy measured within the **last 3 hours** and the measurement differs from the forecast by **more than 30 %**, the forecast wave height for the **next 12 hours** is adjusted step by step towards the measurement. The explanation mentions this.

## 4️⃣ Final score

$$
\text{score} = \alpha \cdot \text{text score} + (1 - \alpha) \cdot CF
$$

| Query type | $\alpha$ | Meaning |
|---|---|---|
| Asks about conditions, time or skill (default) | **0.5** | text and conditions count equally |
| No conditions angle, e.g. *"reef break with showers"* | **0.85** | text counts more |

**Safety rule:** if a spot is unsafe for the user's level (waves too big, or the spot is not meant for that level), **1 point is subtracted**. It drops to the bottom of the list and gets a **warning**.

> 🎛️ **Tuning:** with `TUNE_ALPHA = True` in notebook 05, $\alpha$ is optimised on the dev queries. This is the only place where anything is "learned" from data, and even here the system simply tries the fixed values 0.0, 0.1, …, 1.0 and keeps the best one.